In [ ]:
import torch
import numpy as np
import cv2
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch import Tensor, nn
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Module, Dropout, Flatten, Conv2d, MaxPool2d, AdaptiveAvgPool2d
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from pathlib import Path
import matplotlib.pyplot as plt
import time

# Este projeto roda apenas na GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não encontrada. Verifique se o kernel do notebook é o .venv, "
        f"se o PyTorch é a versão CUDA (instalado: {torch.__version__}) "
        "e se o driver da NVIDIA está instalado (rode nvidia-smi no terminal)."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float().to(device)
        self._y = torch.from_numpy(labels).long().to(device)

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [ ]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()

        self.net = Sequential(
            Conv2d(1, 16, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(16, 32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(32, 64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            AdaptiveAvgPool2d(1),
            Flatten(),
            Dropout(p=0.25),
            Linear(64, output_size),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [ ]:
def train_model_class(features, targets, kf, batch, epochs, criterion, exp=False, lr=0.001):
    print(f"Treinando na GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
    acc_scores = []
    f1_scores = []
    folds_loss = []
    folds_acc = []
    folds_time = []
    num_classes = len(np.unique(targets))
    for i, (train_index, test_index) in enumerate(kf.split(features, targets)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        inicio = time.perf_counter()
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch, shuffle=True)
        model = CustomModel(16900, num_classes).to(device)
        optmizer = Adam(model.parameters(), lr=lr)
        epoch_loss = []
        epoch_acc = []
        for i_epochs in range(epochs):
            model.train()
            batches_loss = []
            batches_acc = []
            for x, y in train_loader:
               # x = x.float().to(device)
               # y = y.to(device)
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()
                batches_loss.append(loss.item())
                batches_acc.append(accuracy_score(y.cpu().numpy(), torch.argmax(output, dim=1).cpu().numpy()))

            epoch_loss.append(np.mean(batches_loss))
            epoch_acc.append(np.mean(batches_acc))

        folds_time.append(time.perf_counter() - inicio)
        folds_loss.append(epoch_loss)
        folds_acc.append(epoch_acc)
        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float().to(device)
            y_true = targets[test_index]
            output = model(x_test_tensor)
            result = torch.argmax(output, dim=1).cpu().numpy()
            acc_s = accuracy_score(y_true, result)
            f1_s = f1_score(y_true, result, average="macro")
            cm = confusion_matrix(y_true, result)
            acc_scores.append(acc_s)
            f1_scores.append(f1_s)
            print(f"Acc: {acc_s}")
            print(f"F1-Score: {f1_s}")
            print(f"CM: {cm}")

        if exp:
            break

    print(f"\nResultado modelo ----- Acc: {np.mean(acc_scores)}, F1-Score: {np.mean(f1_scores)}")

    return folds_loss, folds_acc, folds_time, acc_scores, f1_scores

In [ ]:
def format_time(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


def plot_loss(folds_loss, folds_acc, folds_time, acc_scores, f1_scores, dataset, max_epochs, batch, lr, n_folds, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    folds_acc = np.array(folds_acc) * 100
    for idx, fold_l in enumerate(folds_loss):
        fig, ax1 = plt.subplots(figsize=(8, 3.5))
        x = np.linspace(0, len(fold_l), len(fold_l))
        ax1.plot(x, fold_l, color="red", label="Erro")
        ax1.set_xlabel("Épocas")
        ax1.set_ylabel("Erro")
        ax1.set_title(
            f"{dataset} | Fold {idx + 1}/{n_folds}\n"
            f"Épocas: {max_epochs} | Batch: {batch} | LR: {lr} | Tempo: {format_time(folds_time[idx])}\n"
            f"Teste ----- Acc: {acc_scores[idx] * 100:.2f}% | F1-Score: {f1_scores[idx] * 100:.2f}%"
        )
        # ax1.set_ylim(0, 0.2)
        ax2 = ax1.twinx()
        ax2.plot(x, folds_acc[idx], color="blue", label="Acc")
        ax2.set_ylabel("Acurácia (%)")
        # ax2.set_ylim(0.8, 1.1)
        ax2.grid()
        lines = ax1.get_lines() + ax2.get_lines()
        ax1.legend(lines, [line.get_label() for line in lines], loc="center right")
        fig.tight_layout()
        fig.savefig(output_dir / f"fold_{idx + 1}.png", dpi=150, bbox_inches="tight")
        fig.show()


def plot_summary(folds_loss, folds_acc, folds_time, acc_scores, f1_scores, dataset, max_epochs, batch, lr, n_folds, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    folds_acc = np.array(folds_acc) * 100
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for idx, fold_l in enumerate(folds_loss):
        x = np.linspace(0, len(fold_l), len(fold_l))
        ax1.plot(x, fold_l, alpha=0.6, label=f"Fold {idx + 1} ({format_time(folds_time[idx])})")
        ax2.plot(x, folds_acc[idx], alpha=0.6, label=f"Fold {idx + 1} (Acc: {acc_scores[idx] * 100:.2f}% | F1: {f1_scores[idx] * 100:.2f}%)")

    # Média entre os folds
    ax1.plot(x, np.mean(folds_loss, axis=0), color="black", linewidth=2, label="Média")
    ax2.plot(x, folds_acc.mean(axis=0), color="black", linewidth=2, label="Média")

    ax1.set_xlabel("Épocas")
    ax1.set_ylabel("Erro")
    ax1.set_title("Erro no treino")
    ax1.grid()
    ax1.legend()
    ax2.set_xlabel("Épocas")
    ax2.set_ylabel("Acurácia (%)")
    ax2.set_title("Acurácia no treino (legenda: Acc e F1-Score no teste)")
    ax2.grid()
    ax2.legend()
    fig.suptitle(
        f"{dataset} | Folds: {len(folds_loss)}/{n_folds} | Épocas: {max_epochs} | Batch: {batch} | LR: {lr} | "
        f"Tempo total: {format_time(sum(folds_time))}\n"
        f"Teste ----- Acc média: {np.mean(acc_scores) * 100:.2f}% | F1-Score médio: {np.mean(f1_scores) * 100:.2f}%"
    )
    fig.tight_layout()
    fig.savefig(output_dir / "resumo.png", dpi=150, bbox_inches="tight")
    print(f"Gráficos salvos em: {output_dir}")
    fig.show()

In [ ]:
def load_data(path: str, limit=1000, size=(128, 128)):
    input_dir = Path(path)
    data_array = []

    image_paths = sorted(
        p for p in input_dir.iterdir() if p.is_file()
    )[:limit]

    for image_path in image_paths:
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        image = cv2.resize(image, size)
        data_array.append(image)

    return np.asarray(data_array, dtype=np.uint8)

In [ ]:
# Pasta com as subpastas lung_aca, lung_bnt e lung_scc
# Docker
# DATA_DIR = Path("/app/data/Lung_Cancer")
# VS Code (PC)
DATA_DIR = Path("../data/Lung_Cancer")

class_aca = load_data(DATA_DIR / "lung_aca")
class_bnt = load_data(DATA_DIR / "lung_bnt")
class_scc = load_data(DATA_DIR / "lung_scc")

aca_target = np.zeros(len(class_aca), dtype=int)
bnt_target = np.full(len(class_bnt), 1, dtype=int)
scc_target = np.full(len(class_scc), 2, dtype=int)

features = np.concatenate((class_aca, class_bnt, class_scc))
features = np.expand_dims(features, axis=1)

targets = np.concatenate((aca_target, bnt_target, scc_target))

In [ ]:
print(targets)
print(features.shape)

In [ ]:
dataset = "Pulmão cinza"
criterion = CrossEntropyLoss()
max_epochs = 1500
batch = 64
lr = 0.001
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pasta onde os gráficos são salvos (uma subpasta por execução, para não sobrescrever)
# Docker
# output_dir = Path("/app/graficos") / dataset.replace(" ", "_") / time.strftime("%Y-%m-%d_%H-%M-%S")
# VS Code (PC)
output_dir = Path("../graficos") / dataset.replace(" ", "_") / time.strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
folds_loss, folds_acc, folds_time, acc_scores, f1_scores = train_model_class(features, targets, kf, batch, max_epochs, criterion, True, lr)

In [ ]:
plot_loss(folds_loss, folds_acc, folds_time, acc_scores, f1_scores, dataset, max_epochs, batch, lr, kf.get_n_splits(), output_dir)

In [ ]:
plot_summary(folds_loss, folds_acc, folds_time, acc_scores, f1_scores, dataset, max_epochs, batch, lr, kf.get_n_splits(), output_dir)